# **IoT 보안 기술문서 검색 시스템 (RAG)**

## 프로젝트 목표
1. IoT 보안 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 벡터·키워드 검색과 IoT 보안 답변 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- IoT 보안 RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [91]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://46a1ece4-e238-4def-b529-37daefaa6ba3.us-east-1-1.aws.cloud.qdrant.io:6333


## 1. PDF 문서 로딩

IoT 공통 보안 가이드와 암호·인증 기술 안내서를 불러옵니다.

In [92]:
from langchain_core.documents import Document
import fitz

# 처리할 PDF 파일 목록
pdf_files = [
    "../datasets/IoT_공통보안가이드(최종).pdf",
    "../datasets/사물인터넷(IoT)_환경에서의_암호_인증기술_이용_안내서(17년_개정본).pdf",
]

docs = []

# 각 PDF를 페이지 단위의 Document로 변환 (Parent Document)
for file_path in pdf_files:
    doc = fitz.open(file_path)
    source_name = file_path.split("/")[-1]

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text", sort=True)

        # 빈 페이지는 스킵
        if len(text.strip()) < 10:
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": source_name,
                    "page": page_num + 1,
                    "parent_id": f"{source_name}_page_{page_num + 1}"
                }
            )
        )

    doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 218개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 65자
평균 페이지 길이: 1596자

첫 페이지 내용 미리보기:
   IoT Common Security Guide

ICT 융합 제품·서비스의
보안 내재화를 위한
공통 보안 가이드...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [93]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,      # PDF 후보 비교 평가에서 가장 높은 점수
    chunk_overlap=100    # 긴 보안 문맥과 문장 연결 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 218
  - Child chunk 수: 779
  - 평균 chunk/page: 3.6

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: IoT_공통보안가이드(최종).pdf_page_1
  Page: 1
  Length: 62자
  Content: IoT Common Security Guide

ICT 융합 제품·서비스의
보안 내재화를 위한
공통 보안 가이드...

Chunk 2:
  Parent ID: IoT_공통보안가이드(최종).pdf_page_3
  Page: 3
  Length: 73자
  Content: 1      2016. 9           ICT 융합 제품·서비스의 보안 내재화를 위한 IoT 공통 보안 가이드

2

3

4...

Chunk 3:
  Parent ID: IoT_공통보안가이드(최종).pdf_page_4
  Page: 4
  Length: 641자
  Content: Contents


        I              1. 배경 및 범위 / 8
  개요            2. IoT 보안위협 / 9

                  ...


## 3. Qdrant Cloud에 Child Chunk 저장

IoT 보안 문서 전용 Qdrant 컬렉션을 준비합니다.

In [94]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://46a1ece4-e238-4def-b529-37daefaa6ba3.us-east-1-1.aws.cloud.qdrant.io:6333


In [95]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# IoT 보안 문서 컬렉션
collection_name = "iot_hardware_security_agent"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'iot_hardware_security_agent'이 이미 존재합니다.
컬렉션 'iot_hardware_security_agent' 삭제 중...
컬렉션이 삭제되었습니다.
컬렉션 'iot_hardware_security_agent' 생성 완료

779개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [96]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 218개의 Parent 문서 저장 완료

Docstore 키 예시: ['IoT_공통보안가이드(최종).pdf_page_1', 'IoT_공통보안가이드(최종).pdf_page_3', 'IoT_공통보안가이드(최종).pdf_page_4', 'IoT_공통보안가이드(최종).pdf_page_5', 'IoT_공통보안가이드(최종).pdf_page_6']


## 5. Parent Document Retriever 구현

In [97]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

IoT 보안 질문으로 Child 검색과 Parent 검색 결과를 비교합니다.

In [98]:
# IoT 하드웨어 보안 검색 질문
query = "스마트 도어락을 만들었는데, 기판 설계랑 안에 저장되는 개인정보가 안전한지 어떻게 확인해?"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 스마트 도어락을 만들었는데, 기판 설계랑 안에 저장되는 개인정보가 안전한지 어떻게 확인해?


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 34
  Parent ID: IoT_공통보안가이드(최종).pdf_page_34
  길이: 630자
  내용: ※ 출처: Security and Resilience of Smart Home Environments(“15) – ENSIA



 소프트웨어 보안 기술은 기존의 보안 소프트웨어를 그대로 혹은 기능 축소, 경량화 하여

적용하고자 하는 것이 대부분으로 C1클래스까지는 일부 적용이 가능하나 C0의 경우는 적용이 어려워

소프트웨어 보안 기술만으로는 보안 기능 제공이 힘들다. 하지만 C0의 제품이지만 헬스데이터를

측정하거나, 개인정보 등을 취급, 다루는 IoT 장치는 보안 기술 적용을 해야 하기 때문에 저전력,

저사양에서 구동 가능한 IoT 적용 소프트웨어 보안 기술을 탑재해야 한다.

 하드웨어 보안 기술은 크게 안전한 하드웨어를 활용한 저장 공간 확보와 HW Crypto accelerator와

같이 하드웨어를 이용한 보안 기술 구현으로 앞서, 언급한 C0 클래스의 소프트웨어 보안 기술을

적용하기 힘든 IoT 장치에 보안을 적용하기 위한 단점을 극복할 수 있다는 장점을 가진다.

 일반적으로 하드웨어 보안이라고 하면 Secure Element(이하 SE라 한다.)을 의미하며, SE는 암호 키,

ID, 보안 속성 정보, 인증정보, 금융정보, 서비스 애플리케이션 등 중요 데이터를 안전하게 저장할 수


34

Chunk 2:
  페이지: 83
  Parent ID: IoT_공통보안가이드(최종).pdf_page_83
  길이: 191자
  내용: •수집된 민감 정보 보호를 위해 암호화 적용

  •인가된 자에 한해 개인정보 등 민감 정

## 7. IoT 보안 RAG 답변 시스템 구현

IoT 하드웨어 설계와 보안을 함께 설명하는 시스템 프롬프트를 적용합니다.

In [99]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import Optional
from qdrant_client.http import models

llm = init_chat_model("gpt-5.4-mini")

# 카테고리 분류 결과를 위한 Pydantic 모델
class CategoryClassification(BaseModel):
    """IoT 하드웨어·보안 카테고리 분류 결과"""
    category: Optional[str] = Field(
        description="선택된 카테고리 이름. 적합한 카테고리가 없으면 None"
    )

def determine_category(question: str) -> Optional[str]:
    """
    LLM으로 질문을 분석해 적절한 IoT 하드웨어·보안 카테고리를 결정합니다.

    Args:
        question: 사용자 질문

    Returns:
        카테고리 이름 (문자열) 또는 None (필터 없음)
    """

    # 사용 가능한 카테고리 목록
    available_categories = {
    # ===== 하드웨어 설계 =====
    "회로_전원_신호설계": "전원회로, 전압·전류, 저항·커패시터, 풀업·풀다운, 디커플링, 노이즈, 신호 안정성, 저전력 설계 등 회로 설계 관련",
    "PCB_배선_기판설계": "PCB 구조, 부품 배치, 배선, 통신선 배치, 테스트 포인트, 기판 내층 설계, 개발용 PCB와 양산용 PCB 구성 관련",
    "MCU_메모리_부품설계": "MCU, 메모리, 저장장치, 센서, 주변 IC, 하드웨어 모듈 선택과 연결, 부품 구성 및 하드웨어 구조 관련",
    "인터페이스_통신설계": "UART, JTAG, SPI, I2C, USB, GPIO 등 내부·외부 인터페이스, 센서 연결, 부품 간 통신 구조 및 입출력 설계 관련",


    # ===== 보안 =====
    "하드웨어_물리보안": "제품 분해, 디버그 포트 노출, 내부 회로 접근, 메모리·역공학·부채널 공격, 물리적 조작 및 하드웨어 보호 관련",
    "인증_접근통제": "사용자 인증, 기기 간 인증, 비밀번호, 접근권한, 비인가 사용자나 장치의 접근 차단 관련",
    "암호화_데이터보호": "개인정보와 중요정보 보호, 저장·전송 데이터 암호화, 암호키 관리, 데이터 무결성, 안전한 통신 관련",
    "펌웨어_플랫폼보안": "펌웨어 추출·변조 방지, 소프트웨어 취약점, 안전한 부팅, 보안패치, 안전한 업데이트 및 플랫폼 보호 관련",
}

    # LLM에게 카테고리 분류 요청
    category_list = "\n".join([f"- {cat}: {desc}" for cat, desc in available_categories.items()])

    classification_prompt = f"""다음 질문을 분석하여 가장 적합한 IoT 하드웨어·보안 카테고리를 선택하세요.

<available_categories>
{category_list}
</available_categories>

<question>
{question}
</question>

<rules>
1. 질문의 주요 주제와 가장 관련 있는 카테고리를 선택하세요
2. 여러 카테고리가 관련될 수 있지만, 가장 핵심적인 하나만 선택하세요
3. 적합한 카테고리가 없거나 매우 일반적인 질문이면 category를 null로 설정하세요
</rules>
"""

    # Structured Output을 사용하여 LLM 호출
    structured_llm = llm.with_structured_output(CategoryClassification)
    result = structured_llm.invoke(classification_prompt)

    print(f"[LLM 분류 결과]")
    print(f"  카테고리: {result.category}")

    return result.category


def rag_with_dynamic_filter(question: str) -> str:
    """
    동적 필터링을 적용한 RAG
    """
    # 1. 질문 분석하여 카테고리 결정
    category = determine_category(question)  # 사용자 질문 > 어떤 카테고리인지 LLM에게 물어봄

    # 2. 필터 설정
    search_kwargs = {"k": 3}
    if category:
        search_kwargs["filter"] = models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.category",
                    match=models.MatchValue(value=category)
                )
            ]
        )
        print(f"✓ 적용된 필터: category = '{category}'\n")
    else:
        print(f"✓ 필터 없음 (전체 문서 검색)\n")

    # 3. 문서 검색
    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)  # retriever > invoke
    retrieved_docs = retriever.invoke(question)

    # 4. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        page = doc.metadata['page']
        cat = doc.metadata['category']
        context_parts.append(
            f"[출처: {doc.metadata['source']}, 페이지: {page}, 카테고리: {cat}]\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 5. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return response.content


# IoT 하드웨어·보안 점검 Agent 시스템 프롬프트
template = """
당신은 IoT 디바이스의 하드웨어 설계와 보안을 통합적으로 분석하고 점검하는 전문가입니다.

당신은 홈·가전 IoT 제품을 대상으로 다음 두 영역을 동등하게 분석합니다.

1. 하드웨어 설계 관점 (50%)
   - 기판(PCB) 구조
   - 내부·외부 입출력 포트
   - MCU와 주변 하드웨어 구성
   - 메모리 및 저장장치
   - UART, JTAG, SPI, I2C 등의 인터페이스
   - 개발용 포트와 양산용 하드웨어 구성
   - 주요 부품 간 통신 구조
   - 제품 분해 시 접근 가능한 하드웨어
   - 하드웨어 보안 모듈의 적용 위치와 연결 구조
   - 물리적 조작 및 분해를 고려한 설계

2. 보안 관점 (50%)
   - 개인정보 및 중요정보 보호
   - 인증과 접근통제
   - 암호화
   - 암호키 보호
   - 펌웨어 보호
   - 메모리 및 역공학 공격 대응
   - 디버그 포트를 이용한 공격 대응
   - 물리적 공격 대응
   - 부채널 공격 대응
   - 안전한 업데이트
   - 제품 무단 조작 방지

두 관점 중 하나에 치우치지 말고,
사용자의 질문과 관련된 경우 하드웨어 설계와 보안을 가능한 한 1:1 비율로 함께 분석하세요.

당신의 목표는 단순히 "안전하다", "위험하다"라고 판단하는 것이 아닙니다.

사용자가 제시한 IoT 제품의 구조를 분석하여

- 하드웨어가 어떻게 구성되어 있는지
- 하드웨어 설계에서 어떤 부분을 확인해야 하는지
- 해당 설계가 어떤 보안 위험과 연결되는지
- 공격자가 어떤 부분을 악용할 수 있는지
- 설계를 어떻게 변경하거나 보완하면 좋은지

를 종합적으로 설명하세요.


[중요 원칙]

1. 반드시 아래 [참고 정보]에 있는 내용만 근거로 답하세요.

참고 정보에서 확인할 수 없는 기능, 회로, 부품 사양 또는 보안 기능을
임의로 만들어내지 마세요.

자료만으로 정확한 판단이 어려운 경우에는

"현재 참고 자료만으로는 이 부분을 정확하게 판단하기 어렵습니다."

라고 명확하게 말하세요.


2. 분석은 전문가 수준으로 수행하세요.

단순한 키워드 설명이 아니라
제품의 하드웨어 구조 → 발생 가능한 문제 → 보안 위험 → 개선 방법
사이의 관계를 분석하세요.

예를 들어 사용자가

"스마트 도어락을 만들었는데 내부에 중요한 정보를 저장해도 괜찮아?"

라고 질문하면 단순히

"암호화해야 합니다."

라고 답하지 마세요.

다음 내용을 종합적으로 검토하세요.

[하드웨어 설계]
- 정보가 어떤 저장장치에 저장되는지
- 저장장치에 물리적으로 접근할 수 있는지
- MCU와 저장장치 사이의 통신 구조가 노출되는지
- 개발용 또는 디버그용 포트가 남아 있는지
- 제품을 분해했을 때 주요 부품에 쉽게 접근할 수 있는지

[보안]
- 저장된 중요정보를 읽어갈 수 있는지
- 내부 프로그램이나 펌웨어를 추출할 수 있는지
- 암호화가 필요한지
- 인증되지 않은 접근을 막을 수 있는지
- 중요한 암호키가 안전하게 보호되는지


3. 하드웨어 설계와 보안을 서로 분리된 문제로 보지 마세요.

하드웨어 설계가 보안에 어떤 영향을 주는지 연결해서 설명하세요.

예:

"기판에 개발용 포트를 남겨둠"
→ 외부에서 내부 시스템에 접근할 통로가 생김
→ 펌웨어나 저장정보를 읽을 가능성이 생김
→ 양산 제품에서는 제거·비활성화 또는 접근 제한 필요

이와 같이

[하드웨어 설계]
        ↓
[보안 취약점]
        ↓
[가능한 공격]
        ↓
[발생 가능한 피해]
        ↓
[설계 개선]

순서로 분석하세요.


[하드웨어 설계 분석 기준]

4. 하드웨어 관련 질문에서는 다음 항목을 우선적으로 확인하세요.

- 개발용 PCB와 실제 판매용 PCB의 구성이 적절한지
- UART, JTAG 등 개발·점검용 포트가 제품에 남아 있는지
- 외부에서 접근 가능한 입출력 포트가 있는지
- 중요한 통신선이 쉽게 식별되거나 접근 가능한지
- 테스트 포인트가 외부에서 쉽게 발견되는지
- MCU, 메모리, 보안 관련 부품이 어떻게 연결되는지
- 기기를 분해했을 때 중요 부품에 쉽게 접근할 수 있는지
- 중요한 정보를 일반 저장공간과 분리할 필요가 있는지
- 별도의 하드웨어 보안 모듈을 적용할 필요가 있는지
- 하드웨어 보안 모듈과 MCU 사이의 내부 통신을 보호할 필요가 있는지


5. 하드웨어 설계를 개선할 수 있는 내용이 참고 정보에 있다면
구체적인 설계 방향을 제시하세요.

예를 들어 참고 정보가 뒷받침하는 경우 다음과 같은 내용을 설명할 수 있습니다.

- 개발용 포트를 양산 PCB에서 제거
- 필요 없는 내부 인터페이스 비활성화
- 주요 통신선을 외부에서 쉽게 접근하기 어렵게 구성
- 테스트 포인트의 노출 최소화
- 중요한 데이터와 암호키를 별도의 안전한 하드웨어에 저장
- MCU와 보안 모듈 사이의 통신 보호

단, 참고 정보에 없는 구체적인 수치는 만들지 마세요.


6. 다음과 같은 전자회로 상세 값이 참고 정보에 없다면
임의로 추천하지 마세요.

- 저항값
- 커패시터값
- 정확한 전압값
- 정확한 전류값
- 특정 MCU 핀 번호
- 특정 부품 모델명
- PCB 패턴 폭
- 구체적인 배선 길이
- 상세 회로도

이런 질문을 받으면

"현재 참고 자료에서는 보안과 관련된 하드웨어 설계 방향은 확인할 수 있지만,
구체적인 회로 수치까지는 제공하지 않습니다."

라고 설명하세요.


[보안 분석 기준]

7. 다음 보안 문제를 우선적으로 점검하세요.

- 개발용 또는 디버그 포트 노출
- 펌웨어 및 내부 프로그램 추출
- 메모리 및 저장 데이터 노출
- 개인정보 및 인증정보의 안전하지 않은 저장
- 암호키 노출
- 인증되지 않은 사용자 접근
- 제품의 무단 제어
- 펌웨어 변조
- 제품 분해를 통한 물리적 공격
- 역공학 공격
- 부채널 공격
- 안전하지 않은 업데이트


8. 보안 문제를 발견하면 반드시 다음 내용을 설명하세요.

① 무엇이 문제인지
② 어떤 하드웨어 설계 때문에 문제가 발생하는지
③ 왜 보안상 위험한지
④ 실제로 어떤 일이 발생할 수 있는지
⑤ 하드웨어 또는 보안 설정을 어떻게 개선하면 좋은지


[위험도 판단]

9. 필요한 경우 위험도를 다음과 같이 표시하세요.

[높음]
제품 출시 전에 우선적으로 점검하거나 수정하는 것이 좋은 문제

[중간]
당장 심각하지 않을 수 있지만 개선을 권장하는 문제

[낮음]
위험은 비교적 낮지만 추가 확인이 필요한 문제

참고 정보만으로 위험도를 판단하기 어려우면
임의로 위험도를 지정하지 마세요.


[사용자 설명 원칙]

10. 분석 과정은 전문가 수준으로 수행하지만
최종 답변은 일반 사용자가 이해할 수 있도록 최대한 쉽게 작성하세요.

전문 용어를 먼저 던지지 마세요.

먼저 쉬운 말로 설명하고,
필요할 때 전문 용어를 괄호 안에 표시하세요.


예시 1

나쁜 답변:
"JTAG 인터페이스가 노출되어 펌웨어 덤프 공격에 취약합니다."

좋은 답변:
"기판에 개발자가 내부 프로그램을 확인할 때 사용하는 연결 통로(JTAG)가
그대로 남아 있으면, 기기를 뜯은 사람이 내부 프로그램을 읽어갈 수 있습니다."


예시 2

나쁜 답변:
"암호키를 Secure Element에 저장해야 합니다."

좋은 답변:
"데이터를 잠그는 데 사용하는 중요한 비밀값은 일반 저장공간에 두기보다,
외부에서 쉽게 읽을 수 없도록 별도로 보호된 하드웨어에 저장하는 방법을 고려할 수 있습니다."


예시 3

나쁜 답변:
"PCB 내부 신호선에 대한 물리적 공격 대응이 필요합니다."

좋은 답변:
"기판을 열었을 때 중요한 통신선이 바로 드러나면 공격자가 신호를 분석하기 쉬워집니다.
따라서 중요한 연결 부분을 외부에서 쉽게 찾거나 접근하기 어렵게 설계하는 것이 좋습니다."


11. 전문 용어를 사용해야 한다면 간단히 뜻을 함께 설명하세요.

예:

- PCB: 전자부품들이 연결되어 있는 기판
- MCU: 제품의 동작을 제어하는 핵심 칩
- UART: 기기 내부에서 데이터를 주고받거나 개발 중 상태를 확인하는 통신 통로
- JTAG: 개발자가 칩 내부를 확인하거나 점검할 때 사용하는 연결 통로
- 펌웨어: 기기 안에서 실제로 동작하는 프로그램
- 암호키: 데이터를 잠그고 푸는 데 사용하는 비밀값


[제품별 분석]

12. 사용자가 특정 IoT 제품을 말하면
그 제품의 기능과 다루는 정보를 고려하여 분석하세요.

예:

스마트 도어락
→ 출입 제어, 인증정보, 중요정보 저장, 물리적 접근

홈캠 / 네트워크 카메라
→ 영상정보, 원격접속, 카메라 제어, 내부 저장정보

스마트TV
→ 사용자 계정, 네트워크 연결, 카메라·마이크

공유기 / 게이트웨이
→ 네트워크 접근, 인증정보, 다른 IoT 기기와의 연결

센서 제품
→ 측정정보, 데이터 변조, 물리적 조작


13. 제품 이름만으로 문제가 있다고 단정하지 마세요.

사용자가 설명한 구조와
[참고 정보]에서 검색된 내용을 근거로 판단하세요.


[답변 균형]

14. 질문이 하드웨어와 보안 모두 관련되어 있다면
답변의 비중을 가능한 한 다음과 같이 유지하세요.

하드웨어 설계 분석 : 약 50%
보안 분석 : 약 50%

보안 내용만 길게 설명하거나,
반대로 하드웨어 구조만 설명하지 마세요.

두 내용을 반드시 연결하여 최종적인 개선 방향을 제시하세요.


[답변 형식]

### 종합 점검 결과

제품에서 가장 중요하게 확인해야 할 내용을
1~2문장으로 먼저 설명하세요.


### 하드웨어 설계 점검

다음을 중심으로 설명하세요.

- 현재 설계에서 확인해야 할 부분
- 문제가 될 수 있는 하드웨어 구조
- 설계상 개선할 수 있는 부분


### 보안 점검

다음을 중심으로 설명하세요.

- 해당 하드웨어 구조가 왜 위험할 수 있는지
- 어떤 공격이나 정보 유출로 이어질 수 있는지
- 필요한 보호 방법


### 통합 개선안

하드웨어 설계와 보안을 함께 고려하여
실제로 어떻게 개선하는 것이 좋은지 설명하세요.

가장 중요한 조치부터 순서대로 제시하세요.


### 참고 자료

실제로 답변에 사용한 문서명과 페이지 번호만 표시하세요.

예:
- 홈·가전 IoT 보안가이드, p.42
- 홈·가전 IoT 보안가이드, p.142


[참고 정보]
{context}


[사용자 질문]
{question}


[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content


print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. IoT 하드웨어·보안 질문 테스트

서로 다른 IoT 기기와 보안 상황으로 RAG 답변을 테스트합니다.

In [100]:
# IoT 하드웨어·보안 기능별 테스트 질문
questions = [
    "우리 집 홈캠의 계정, 업데이트, 저장정보, 외부 포트를 기준으로 보안 상태를 진단해줘.",

    "홈캠이 혼자 움직이고 모르는 로그인 기록이 있어. 지금 할 일과 확인할 것을 알려줘.",

    "스마트 도어락의 JTAG와 UART가 양산 기판에 남아 있어. 위험한 이유와 개선 방법을 점검해줘.",

    "스마트 플러그를 양산하기 전에 확인할 하드웨어 보안 체크리스트와 점검 기준을 만들어줘.",

    "성능과 전력이 제한된 IoT 센서에서 MCU와 보안칩을 연결하고 인증정보를 보호할 방법을 추천해줘.",

    "기본 비밀번호, 열린 디버그 포트, 오래된 펌웨어가 발견됐어. 무엇부터 고칠지 이유와 함께 정리해줘."
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 우리 집 홈캠의 계정, 업데이트, 저장정보, 외부 포트를 기준으로 보안 상태를 진단해줘.



### 종합 점검 결과

현재 참고 자료만으로는 **특정 홈캠 제품의 실제 PCB 구성이나 포트 배치까지는 확인할 수 없습니다.**  
다만, 홈캠은 **계정 정보, 업데이트 경로, 저장정보, 외부 포트**가 공격 표면(공격자가 노릴 수 있는 접점)이 되기 쉬우므로, 이 4가지를 중심으로 **디버그 포트 노출, 펌웨어 분석, 저장 메모리 유출, 업데이트 변조** 가능성을 함께 점검해야 합니다.

---

### 하드웨어 설계 점검

#### 1) 계정과 저장정보를 어디에 두는지 확인
- 홈캠은 영상, 계정 정보, 인증 정보, 장치 설정값을 저장할 수 있습니다.
- 이 정보가 **일반 저장공간에 그대로 남아 있으면**, 분해 후 저장장치 접근이나 메모리 분석을 통해 읽힐 수 있습니다.
- 참고 자료에서는 IoT 장치 모의해킹 시 **장치 메모리, 펌웨어, 하드코딩된 설정, 암호화 키 노출**을 함께 점검하라고 안내합니다.  
  → 즉, 저장정보가 펌웨어나 메모리 안에 같이 들어 있는지 확인해야 합니다.

#### 2) 업데이트 경로가 하드웨어적으로 어떻게 연결되는지 확인
- 홈캠의 업데이트는 단순 소프트웨어 문제가 아니라, **외부 네트워크와 기기 내부 프로그램이 연결되는 경로**입니다.
- 업데이트 파일 검증이 약하거나, 기기 내부에 업데이트 검증용 정보가 노출되면 공격자가 이를 악용할 수 있습니다.
- 참고 자료에는 **암호화 미적용 업데이트**, **펌웨어 버전 출력**, **펌웨어 변조를 통한 영구 장악**이 모의해킹 시나리오에 포함되어 있습니다.  
  → 따라서 업데이트 기능은 외부에서 쉽게 조작되지 않도록 확인해야 합니다.

#### 3) 외부 포트와 디버그 통로 확인
- 홈캠을 분해했을 때 **UART, JTAG 같은 개발·점검용 포트**나 테스트 포인트가 남아 있으면 내부 프로그램과 상태 정보를 읽을 수 있는 통로가 생깁니다.
- 참고 자료는 **디버그 포트**를 공격 표면으로 직접 언급하고 있습니다.  
  → 양산 제품에서 이런 포트가 노출돼 있는지, 아니면 비활성화되었는지 확인이 필요합니다.

#### 4) 내부 부품 접근성 확인
- 기기를 열었을 때 MCU, 저장장치, 통신선이 쉽게 보이면 역공학과 물리적 공격이 쉬워집니다.
- 특히 저장 메모리나 펌웨어 저장부가 분리 보호되지 않으면, 영상·계정·설정값이 추출될 수 있습니다.
- 현재 참고 자료만으로는 홈캠의 실제 하드웨어 배치를 알 수 없으므로, **분해 시 접근 가능한 부품과 포트의 유무를 직접 확인해야 합니다.**

---

### 보안 점검

#### 1) 계정 보호 상태
**문제:** 계정 정보와 인증 정보가 안전하게 보호되지 않으면, 공격자가 홈캠을 원격으로 제어할 수 있습니다.  
**하드웨어와의 연결:** 계정 정보가 메모리나 펌웨어에 평문 또는 약한 방식으로 저장되면 분해나 덤프 공격으로 노출될 수 있습니다.  
**위험:** 타인이 영상 확인, 카메라 제어, 설정 변경을 할 수 있습니다.  
**개선:** 계정 관련 정보는 일반 저장공간에 그대로 두지 말고, 보호된 저장 방식과 강한 인증 절차를 사용해야 합니다.

#### 2) 업데이트 보안
**문제:** 업데이트가 검증되지 않으면 악성 펌웨어로 바뀔 수 있습니다.  
**하드웨어와의 연결:** 업데이트 검증 로직이 MCU 안에 있더라도, 디버그 포트나 펌웨어 추출이 가능하면 우회 분석이 쉬워집니다.  
**위험:** 공격자가 펌웨어를 변조해 장치를 장악하거나, 취약점을 영구적으로 남길 수 있습니다.  
**개선:** 업데이트 파일의 무결성 확인, 인증된 업데이트만 허용, 디버그 경로 최소화가 필요합니다.

#### 3) 저장정보 보호
**문제:** 영상이나 설정값이 저장장치에 그대로 남아 있으면 유출 위험이 큽니다.  
**하드웨어와의 연결:** 저장장치가 외부에서 접근 가능한 위치에 있거나, 기기 분해 시 바로 보이면 물리적 추출이 쉬워집니다.  
**위험:** 사생활 영상, Wi‑Fi 설정, 장치 등록 정보가 노출될 수 있습니다.  
**개선:** 중요한 저장정보는 암호화하거나, 최소한 일반 데이터와 분리해 보호하는 것이 좋습니다.

#### 4) 외부 포트 노출
**문제:** 외부 포트나 내부 디버그 포트가 남아 있으면 공격 진입점이 됩니다.  
**하드웨어와의 연결:** UART, JTAG, 테스트 포인트가 외부에서 쉽게 찾을 수 있으면 내부 정보 접근이 쉬워집니다.  
**위험:** 펌웨어 분석, 설정값 읽기, 디버그 기능 악용이 가능합니다.  
**개선:** 양산 PCB에서는 불필요한 포트를 제거하거나 비활성화하고, 테스트 포인트 노출을 줄여야 합니다.

---

### 통합 개선안

1. **양산용 PCB에서 디버그 포트와 테스트 포인트 노출 여부를 먼저 점검**
   - UART, JTAG 같은 개발용 통로가 남아 있으면 우선 차단하거나 비활성화하는 것이 좋습니다.

2. **계정 정보와 저장정보를 일반 저장공간과 분리해서 보호**
   - 영상, 인증 정보, 설정값이 한곳에 평문으로 남지 않게 해야 합니다.
   - 참고 자료에서 언급된 것처럼, 암호화 키 노출과 하드코딩된 설정도 함께 점검해야 합니다.

3. **업데이트는 인증된 파일만 적용되도록 설계**
   - 암호화 미적용 업데이트나 펌웨어 변조 가능성을 줄여야 합니다.
   - 펌웨어 버전 출력이 공격에 도움이 될 수 있으므로, 불필요한 정보 노출도 줄이는 것이 좋습니다.

4. **분해를 고려한 물리적 방어 강화**
   - 기기를 열었을 때 중요한 부품과 통신선이 바로 드러나지 않도록 배치와 접근성을 점검해야 합니다.
   - 현재 참고 자료만으로 구체적인 부품 위치는 알 수 없으므로, 실물 분해 확인이 필요합니다.

5. **모의해킹 기반 점검 수행**
   - 참고 자료는 정형화된 도구만으로는 다양한 공격 시나리오를 다 보기 어렵다고 설명합니다.
   - 따라서 계정, 업데이트, 저장정보, 디버그 포트를 대상으로 실제 공격 시나리오를 세워 점검하는 것이 적절합니다.

---

### 참고 자료

- IoT_공통보안가이드(최종).pdf, p.58
- IoT_공통보안가이드(최종).pdf, p.115


질문: 홈캠이 혼자 움직이고 모르는 로그인 기록이 있어. 지금 할 일과 확인할 것을 알려줘.



### 종합 점검 결과

지금 상황은 **“누군가 원격으로 홈캠을 조작했거나, 계정이 탈취되었거나, 기기 내부 접근 통로가 남아 있었을 가능성”**을 먼저 의심해야 합니다.  
현재 참고 자료만으로는 이 홈캠의 내부 하드웨어 구조를 직접 확인할 수 없지만, **로그인 기록 이상 + 혼자 움직임**은 보안 관점에서 즉시 확인이 필요한 신호입니다.

---

### 하드웨어 설계 점검

현재 참고 자료에는 이 홈캠의 **PCB 구조, MCU, 저장장치, UART/JTAG 같은 디버그 포트, 내부 포트 노출 여부**가 나오지 않습니다.  
그래서 **기판에 개발용 포트가 남아 있는지, 분해했을 때 주요 부품에 쉽게 접근 가능한지, 저장장치가 따로 있는지**는 지금 자료만으로는 정확히 판단하기 어렵습니다.

다만 홈캠은 보통 다음 하드웨어 구조가 보안에 직접 영향을 줍니다.

- **기판(PCB)와 외부 접근성**
  - 제품을 분해했을 때 중요한 칩이나 저장장치가 바로 보이면, 공격자가 내부 정보에 접근하기 쉬워집니다.
- **내부 통신선과 개발 포트**
  - UART, JTAG 같은 개발·점검용 연결 통로가 남아 있으면, 내부 상태 확인이나 펌웨어 추출로 이어질 수 있습니다.
- **저장장치와 펌웨어**
  - 계정 정보, 인증 정보, 녹화 데이터, 설정값이 일반 저장공간에 있으면 물리적 접근 시 노출 위험이 커집니다.
- **보안 모듈 적용 여부**
  - 암호키를 일반 MCU나 일반 메모리에만 두면, 분해나 역공학에 더 취약할 수 있습니다.

즉, 지금 증상은 단순 앱 문제만이 아니라 **기기 자체가 외부에서 제어될 수 있는 구조인지**도 함께 확인해야 합니다.

---

### 보안 점검

#### 1) 무엇이 문제인지
- **모르는 로그인 기록**
  - 인증되지 않은 접근이 있었을 가능성이 있습니다.
- **홈캠이 혼자 움직임**
  - 기기 제어 권한이 탈취되었거나, 내부 설정이 변조되었을 가능성이 있습니다.

#### 2) 어떤 하드웨어 설계 때문에 문제가 발생할 수 있는지
- 기판에 **디버그 포트(UART, JTAG 등)** 가 남아 있으면 내부 제어 정보가 노출될 수 있습니다.
- 내부 저장장치에 **계정 정보나 인증 정보**가 평문 또는 약하게 보호된 상태로 있으면 탈취 위험이 커집니다.
- 펌웨어 보호가 약하면 **내부 프로그램 추출이나 변조**가 가능해질 수 있습니다.

#### 3) 왜 보안상 위험한지
- 공격자가 카메라의 **실시간 영상, 음성, 저장 영상, 계정 정보**에 접근할 수 있습니다.
- 카메라 방향을 바꾸거나 끄는 식으로 **무단 제어**가 가능해질 수 있습니다.
- 저장된 정보가 유출되면 집 안 상황이 외부에 노출될 수 있습니다.

#### 4) 실제로 어떤 일이 발생할 수 있는지
- 카메라가 원하지 않는 방향으로 움직임
- 낯선 기기나 계정에서 로그인됨
- 영상이 외부로 유출됨
- 계정 비밀번호가 바뀌거나, 기기가 다른 사람에게 묶임
- 펌웨어가 변조되어 계속 재침해될 수 있음

#### 5) 어떻게 개선하면 좋은지
참고 자료에 따르면 다음 보호 방식이 유효합니다.

- **사용자인증**
  - 가능한 다중 요소 인증을 사용해, 계정 탈취만으로 접근되지 않게 해야 합니다.
- **메시지인증**
  - 기기와 서버 사이 명령이 진짜인지 확인해, 가짜 명령이나 변조를 막아야 합니다.
- **채널암호화 / 데이터 암호화**
  - 전송 중 데이터와 저장 데이터를 암호화해 도청과 정보 유출을 줄여야 합니다.
- **소프트웨어 서명**
  - 서명된 펌웨어만 동작하게 해서 변조된 펌웨어 실행을 막아야 합니다.
- **원격 잠금 / 원격 지우기**
  - 이상 징후가 있으면 원격 제어를 제한하고 필요 시 데이터를 삭제할 수 있어야 합니다.
- **로그 분석**
  - 누가 언제 접근했는지 추적해야 합니다.

---

### 지금 할 일과 확인할 것

#### 바로 지금 할 일
1. **홈캠을 인터넷에서 분리**
   - 가능하면 Wi-Fi를 끄거나 네트워크를 잠시 차단하세요.
2. **앱/웹 계정 비밀번호 변경**
   - 홈캠 계정, 이메일 계정, 연결된 클라우드 계정 비밀번호를 바꾸세요.
3. **다중 요소 인증 활성화**
   - 지원된다면 바로 켜세요.
4. **접속 기기 목록 확인**
   - 낯선 로그인 기기, 위치, 시간대를 확인하세요.
5. **카메라 물리적 상태 확인**
   - 카메라 방향이 실제로 움직이는지, LED/작동 표시가 비정상인지 확인하세요.
6. **펌웨어 업데이트 여부 확인**
   - 다만, 이미 침해가 의심되면 업데이트 전에 제조사 공지를 확인하는 것이 좋습니다.
7. **중요한 영상/설정 백업 후 초기화 고려**
   - 이상 징후가 계속되면 출하 시 상태 재설정을 검토하세요.

#### 꼭 확인할 것
- 최근 로그인 시간과 IP/기기 정보
- 카메라가 자동 회전/추적 기능을 원래 지원하는지
- 가족/공유 계정이 있는지
- 다른 앱이나 연동 서비스에서 같은 비밀번호를 쓰는지
- 제조사 앱에서 원격 접속 권한이 누구에게 열려 있는지
- 펌웨어가 최신인지
- 초기화 후에도 같은 증상이 재발하는지

---

### 통합 개선안

가장 중요한 순서대로 보면 다음과 같습니다.

1. **즉시 네트워크 차단**
   - 공격자가 계속 제어하지 못하게 막는 것이 우선입니다.
2. **계정 전면 점검**
   - 홈캠 계정, 이메일, 연동 계정 비밀번호를 모두 변경하고, 다중 요소 인증을 켭니다.
3. **로그 확인**
   - 모르는 로그인 기록이 언제, 어떤 기기에서 발생했는지 확인합니다.
4. **기기 초기화 및 재등록**
   - 침해가 의심되면 초기화 후 다시 설정합니다.
5. **펌웨어 업데이트**
   - 업데이트는 반드시 공식 경로로만 진행합니다.
6. **장기적으로는 보안 기능 강화**
   - 인증, 메시지 인증, 통신 암호화, 서명 검증, 로그 분석 같은 기능이 필요합니다.
7. **하드웨어 관점 점검**
   - 분해 가능한 제품이라면 개발용 포트, 테스트 포인트, 저장장치 접근성을 제조 단계에서 줄여야 합니다.

현재는 하드웨어 내부 구조를 확인할 수 없으므로, **“디버그 포트가 남아 있어서 그런지” 여부는 단정할 수 없습니다.**  
하지만 증상상 **계정 탈취 + 원격 제어 + 기기 변조 가능성**을 함께 보고 대응하는 것이 맞습니다.

---

### 참고 자료

- IoT_공통보안가이드(최종).pdf, p.98


질문: 스마트 도어락의 JTAG와 UART가 양산 기판에 남아 있어. 위험한 이유와 개선 방법을 점검해줘.



...

### 종합 점검 결과

스마트 플러그는 전원 제어 기능 자체보다도, **양산 PCB에 개발용 포트나 점검용 설정이 남아 있는지**와 **펌웨어 업데이트를 안전하게 할 수 있는지**를 가장 먼저 확인해야 합니다.  
현재 참고 자료는 소프트웨어/펌웨어 취약점 점검과 패치 체계를 중심으로 설명하고 있어, **스마트 플러그 전용의 구체적 회로 구성이나 부품 구조까지는 확인되지 않습니다.**

---

### 하드웨어 설계 점검

스마트 플러그의 경우, 양산 전에 다음 항목을 우선 확인하는 체크리스트로 볼 수 있습니다.

#### 1) 양산 PCB에 개발용 포트가 남아 있는지
- **확인할 부분**
  - UART, JTAG 같은 개발·점검용 연결 통로가 양산 제품에도 그대로 노출되어 있는지 확인해야 합니다.
  - 테스트 포인트가 외부에서 쉽게 찾을 수 있는지도 봐야 합니다.
- **문제가 될 수 있는 구조**
  - 개발 단계에서 사용하던 포트가 그대로 남아 있으면, 제품을 분해한 사람이 내부 동작을 확인하거나 프로그램 접근을 시도할 수 있습니다.
- **개선 방향**
  - 양산용 PCB에서는 불필요한 개발용 포트를 제거하거나 비활성화하고, 꼭 필요한 경우에도 외부에서 쉽게 접근하기 어렵게 구성하는 것이 좋습니다.

#### 2) MCU와 주변 부품의 통신 구조가 단순하게 드러나는지
- **확인할 부분**
  - 제품을 열었을 때 MCU(제품 동작을 제어하는 핵심 칩), 저장장치, 주변 회로가 쉽게 접근 가능한지 확인합니다.
  - 주요 통신선이 보드 표면에서 바로 식별되는지도 봐야 합니다.
- **문제가 될 수 있는 구조**
  - 중요한 연결이 노출되어 있으면, 공격자가 신호를 추적하거나 내부 동작을 분석하기 쉬워집니다.
- **개선 방향**
  - 중요한 통신선과 점검 지점을 외부에서 쉽게 찾기 어렵게 하고, 분해했을 때도 바로 확인되지 않도록 설계를 보완하는 것이 좋습니다.

#### 3) 저장장치와 펌웨어 구성 확인
- **확인할 부분**
  - 펌웨어가 일반 저장공간에 저장되는지, 업데이트용 저장공간이 별도로 있는지 확인해야 합니다.
  - 제품 출시 후 업데이트를 안전하게 적용할 수 있는 구조인지 점검합니다.
- **문제가 될 수 있는 구조**
  - 업데이트 기능이 없거나, 업데이트 검증이 약하면 취약점이 발견된 뒤에도 수정이 어렵습니다.
- **개선 방향**
  - 참고 자료에서 강조하듯이, 출시 후 취약점 발견에 대비해 **펌웨어 업데이트 기능**을 구현해야 합니다.
  - 업데이트 과정에서 보안 패치를 안정적으로 배포하고 적용할 수 있어야 합니다.

#### 4) 제품 분해 시 접근 가능한 하드웨어 범위
- **확인할 부분**
  - 기기를 열었을 때 중요한 회로에 쉽게 접근할 수 있는지 확인합니다.
- **문제가 될 수 있는 구조**
  - 분해가 쉬우면 개발용 포트, 테스트 패드, 통신선에 대한 물리적 접근이 쉬워집니다.
- **개선 방향**
  - 물리적 조작을 고려해, 보드 상의 중요한 접점을 최소화하고 외부 노출을 줄이는 방향이 좋습니다.

---

### 보안 점검

참고 자료를 기준으로 볼 때, 스마트 플러그에서 특히 중요한 보안 문제는 **취약점 점검 부족**과 **안전하지 않은 패치 적용**입니다.

#### 1) 개발용 포트 노출로 인한 보안 위험
- **무엇이 문제인지**
  - 양산 제품에 UART, JTAG 같은 개발용 통로가 남아 있으면 내부 접근 경로가 생깁니다.
- **어떤 설계 때문에 생기는지**
  - 개발 편의를 위해 남겨둔 포트나 테스트 포인트가 양산 PCB에 그대로 들어간 경우입니다.
- **왜 위험한지**
  - 공격자가 제품을 분해해 내부 프로그램 확인이나 설정 확인을 시도할 수 있습니다.
- **실제로 어떤 일이 생길 수 있는지**
  - 내부 동작 분석, 취약점 탐색, 설정 오용 가능성이 커집니다.
- **개선 방법**
  - 양산 전 포트 제거 또는 비활성화, 접근 제한, 테스트 포인트 최소화가 필요합니다.

#### 2) 펌웨어 취약점이 남아 있는 위험
- **무엇이 문제인지**
  - 펌웨어나 관련 소프트웨어에 알려진 취약점이 남아 있을 수 있습니다.
- **어떤 설계 때문에 생기는지**
  - 오픈소스, 패키지, API, 펌웨어, OS 등에 대한 취약점 점검이 부족한 경우입니다.
- **왜 위험한지**
  - 알려진 취약점은 공격자에게 표적이 되기 쉽습니다.
- **실제로 어떤 일이 생길 수 있는지**
  - 제품 제어 기능 악용, 비정상 동작, 서비스 장애가 발생할 수 있습니다.
- **개선 방법**
  - 개발 단계에서 보안 취약점 점검을 수행하고, 출시 전에는 알려진 취약점 검사, 소스 코드 검사, 퍼징(Fuzzing) 등의 테스트를 하는 것이 좋습니다.

#### 3) 업데이트 기능이 미흡한 위험
- **무엇이 문제인지**
  - 취약점이 발견된 뒤 이를 수정할 수단이 부족하면 오래된 취약점이 남습니다.
- **어떤 설계 때문에 생기는지**
  - 펌웨어 업데이트 기능이 없거나, 안전하게 배포·적용하는 체계가 없는 경우입니다.
- **왜 위험한지**
  - 취약점을 수정하지 못하면 공격자가 계속 같은 약점을 이용할 수 있습니다.
- **실제로 어떤 일이 생길 수 있는지**
  - 동일한 취약점을 가진 기기가 널리 방치되어 대량 악용될 수 있습니다.
- **개선 방법**
  - 출시 후 취약점 발견에 대비해 소프트웨어(펌웨어) 업데이트 기능을 구현하고, 이용자에게 신속히 적용을 요청할 수 있어야 합니다.

#### 4) 외부 소프트웨어 사용에 따른 위험
- **무엇이 문제인지**
  - 오픈소스나 외부 패키지에 알려진 취약점이 있을 수 있습니다.
- **어떤 설계 때문에 생기는지**
  - 외부 소프트웨어를 사용할 때 취약점 확인이 부족한 경우입니다.
- **왜 위험한지**
  - 공개된 취약점은 공격자가 쉽게 악용할 수 있습니다.
- **실제로 어떤 일이 생길 수 있는지**
  - 네트워크를 통한 공격, 서비스 중단, 장치 오동작으로 이어질 수 있습니다.
- **개선 방법**
  - 사용 중인 외부 소프트웨어의 취약점을 지속적으로 모니터링해야 합니다.

---

### 스마트 플러그 양산 전 하드웨어 보안 체크리스트

아래 항목을 점검 기준으로 사용하면 됩니다.

#### A. PCB/포트 점검
- [ ] UART, JTAG 같은 개발용 포트가 양산 PCB에 남아 있지 않은가
- [ ] 테스트 포인트가 외부에서 쉽게 발견되지 않는가
- [ ] 분해했을 때 내부 점검용 연결부가 바로 드러나지 않는가
- [ ] 불필요한 내부 인터페이스가 양산품에서 제거 또는 비활성화되었는가

#### B. 펌웨어/소프트웨어 점검
- [ ] 펌웨어와 관련 소프트웨어의 알려진 취약점을 확인했는가
- [ ] 출시 전 알려진 취약점 검사, 소스 코드 검사, 퍼징 테스트를 수행했는가
- [ ] 오픈소스/외부 패키지의 취약점 정보를 지속적으로 확인할 수 있는가

#### C. 업데이트 체계 점검
- [ ] 출시 후 취약점 대응을 위한 펌웨어 업데이트 기능이 있는가
- [ ] 안전하고 안정적으로 업데이트를 배포할 수 있는가
- [ ] 이용자에게 업데이트 적용을 요청할 절차가 있는가

#### D. 물리적 분해 대응 점검
- [ ] 제품을 분해했을 때 주요 부품에 쉽게 접근되지 않도록 고려했는가
- [ ] 내부 회로와 통신선이 외부에서 쉽게 식별되지 않도록 설계했는가

---

### 통합 개선안

가장 먼저 할 일은 **양산 PCB에서 개발용 포트를 정리하고, 테스트 포인트와 주요 통신선의 노출을 줄이는 것**입니다. 이것이 안 되어 있으면 물리적 접근만으로도 내부 분석 통로가 열릴 수 있습니다.

그 다음으로는 **출시 전 취약점 점검 체계**를 넣어야 합니다. 참고 자료에서 제시한 것처럼 알려진 취약점 검사, 소스 코드 검사, 퍼징 같은 테스트를 통해 잔류 취약점을 찾아야 합니다.

마지막으로는 **안전한 펌웨어 업데이트 기능**을 준비해야 합니다. 스마트 플러그는 장기간 사용되는 제품이므로, 출시 이후 취약점이 발견되더라도 수정할 수 있어야 합니다.  
즉, 하드웨어는 “열기 어렵게, 찾기 어렵게”, 소프트웨어는 “점검하고, 고치고, 업데이트할 수 있게” 만드는 방향이 좋습니다.

---

### 참고 자료

- IoT_공통보안가이드(최종).pdf, p.55
- IoT_공통보안가이드(최종).pdf, p.99


질문: 성능과 전력이 제한된 IoT 센서에서 MCU와 보안칩을 연결하고 인증정보를 보호할 방법을 추천해줘.



### 종합 점검 결과

성능과 전력이 제한된 IoT 센서에서는 **MCU(제품의 동작을 제어하는 핵심 칩)와 보안칩(중요한 비밀값을 보호하는 별도 하드웨어)**을 어떻게 연결하느냐가 핵심입니다.  
참고 자료 기준으로는, **인증정보(키, 인증서 등)는 일반 저장공간에 두기보다 보호 기능이 있는 별도 하드웨어에 두고**, 장치 인증은 단순 식별자만이 아니라 **인증서·토큰 기반 또는 경량 인증코드 기반**으로 설계하는 방향이 적절합니다.

### 하드웨어 설계 점검

- **현재 설계에서 확인해야 할 부분**
  - MCU가 센서 동작과 통신을 담당하고, 인증정보는 별도 보안칩에 저장하는 구조인지 확인해야 합니다.
  - MCU와 보안칩 사이의 내부 통신 경로가 외부에서 쉽게 접근되거나 추적 가능하지 않은지 점검해야 합니다.
  - 양산 제품에서 개발용 포트나 점검용 인터페이스가 남아 있지 않은지도 확인이 필요합니다. 이런 포트가 남아 있으면 내부 동작 확인 통로가 생길 수 있습니다.

- **문제가 될 수 있는 하드웨어 구조**
  - 인증정보를 MCU 내부 일반 메모리나 외부 저장공간에 넣는 구조는 외부 접근 시 노출 위험이 커집니다.
  - MCU와 보안칩 사이 연결이 단순하고 분리 보호가 약하면, 분해 후 신호를 따라가며 통신 내용을 분석할 가능성이 생깁니다.
  - 개발용 포트가 양산 보드에 그대로 남아 있으면, 내부 상태 확인이나 프로그램 추출의 통로가 될 수 있습니다.

- **설계상 개선할 수 있는 부분**
  - 인증정보는 **별도 보호 기능을 가진 하드웨어**에 저장하고, MCU는 그 값을 직접 읽기보다 필요한 인증 동작만 요청하는 구조를 고려하는 것이 좋습니다.
  - 장치 인증은 **고유 식별자만으로 처리하지 말고**, 인증서·토큰 기반 또는 경량 인증코드 방식으로 보완하는 것이 좋습니다.
  - 양산 PCB에서는 개발용 포트를 제거하거나 비활성화해, 불필요한 접근 통로를 줄이는 방향이 필요합니다.

### 보안 점검

- **해당 하드웨어 구조가 왜 위험할 수 있는지**
  - 참고 자료에 따르면, IoT 장치 인증은 단순한 고유 식별자만 쓰면 복제나 재생공격에 취약할 수 있습니다.
  - 인증정보를 보호하지 않으면, 장치 복제나 오남용을 막기 어렵습니다.
  - 개발·점검용 연결 통로가 남아 있으면 내부 프로그램이나 비밀정보 접근 가능성이 커집니다.

- **어떤 공격이나 정보 유출로 이어질 수 있는지**
  - 공격자가 인증정보를 읽어가면 장치를 정당한 기기처럼 가장할 수 있습니다.
  - 인증 방식이 약하면, 복제된 장치가 네트워크에 들어오는 문제가 생길 수 있습니다.
  - 내부 저장정보나 인증토큰이 노출되면 장치 오남용으로 이어질 수 있습니다.

- **필요한 보호 방법**
  - 인증정보는 보호 기능이 있는 별도 하드웨어에 두어야 합니다.
  - 장치 인증은 인증서·토큰 기반 방식 또는 별도 경량 인증코드 방식으로 구현하는 것이 좋습니다.
  - 가능한 범위에서 사용자나 서비스에 대한 인증 절차를 분리하고, 중복 발급되지 않게 관리해야 합니다.
  - 필요하다면 재인증과 더 세분화된 접근제어도 고려할 수 있습니다.

### 통합 개선안

1. **인증정보를 MCU 일반 메모리에 두지 말고, 보호 기능이 있는 보안칩에 저장**
   - 하드웨어 측면에서 가장 먼저 확인할 부분입니다.
   - 보안 측면에서는 비밀값 유출과 장치 복제를 줄이는 핵심 조치입니다.

2. **MCU는 센서 제어와 통신만 맡고, 인증은 보안칩이 수행하도록 역할 분리**
   - MCU가 모든 비밀값을 직접 다루지 않게 하면 노출 범위를 줄일 수 있습니다.

3. **장치 인증은 고유 식별자만 쓰지 말고 인증서·토큰 또는 경량 인증코드 기반으로 보완**
   - 참고 자료에 따르면 단순 식별자는 복제와 재생공격에 취약할 수 있습니다.

4. **양산 PCB에서 개발용 포트와 불필요한 점검 통로 정리**
   - 분해 시 접근 가능한 부분을 줄이면 물리적 공격 가능성도 낮아집니다.

5. **성능·전력 제약을 고려해 경량화된 인증 방식 선택**
   - 참고 자료는 IoT 환경에서 보안 서비스를 경량화하여 구현해야 한다고 설명합니다.
   - 따라서 센서처럼 자원이 제한된 장치에는 무거운 방식보다 경량 인증 구조가 적합합니다.

### 참고 자료

- IoT_공통보안가이드(최종).pdf, p.21
- IoT_공통보안가이드(최종).pdf, p.32


질문: 기본 비밀번호, 열린 디버그 포트, 오래된 펌웨어가 발견됐어. 무엇부터 고칠지 이유와 함께 정리해줘.



### 종합 점검 결과

가장 먼저 고쳐야 할 것은 **열린 디버그 포트**와 **기본 비밀번호**입니다. 이 두 가지는 기기 내부로 직접 들어가는 통로가 되기 쉬워서, 공격자가 펌웨어나 설정을 건드릴 가능성을 크게 높입니다. 그 다음으로는 **오래된 펌웨어**를 안전하게 갱신하고, 갱신 사실을 사용자가 알 수 있게 하는 것이 중요합니다.

### 하드웨어 설계 점검

- **열린 디버그 포트**
  - 기판에 디버그용 연결 통로가 그대로 남아 있으면, 분해한 사람이 내부 동작을 확인하거나 점검용 기능을 악용할 수 있습니다.
  - 참고 자료에서도 IoT 장치 모의해킹 시 **디버그 포트**, **장치 메모리**, **펌웨어**가 주요 점검 대상로 제시됩니다.
  - 즉, 디버그 포트는 단순한 개발 편의 기능이 아니라, 양산 제품에서는 공격 표면(Attack Surface)이 될 수 있습니다.

- **기본 비밀번호**
  - 기본 비밀번호는 제품 등록, 인증, 권한 검증과 직접 연결됩니다.
  - 하드웨어 자체가 네트워크에 연결되어 있으면, 비밀번호가 약하거나 그대로 남아 있을 때 외부 접근이 쉬워집니다.
  - 이런 경우 기기를 물리적으로 뜯지 않아도 원격에서 접근할 수 있어 위험합니다.

- **오래된 펌웨어**
  - 참고 자료는 보안 패치에서 **주기적 패치 기능**, **암호화된 연결을 통한 패치 전송**, **서명 및 검증**, **가능한 한 보안 부팅 구현**을 제시합니다.
  - 따라서 오래된 펌웨어가 남아 있다는 것은, 이미 알려진 취약점이 그대로 남아 있을 수 있다는 뜻입니다.
  - 하드웨어 측면에서는 펌웨어 업데이트 경로가 제대로 관리되는지, 업데이트 무결성을 확인할 수 있는지 확인해야 합니다.

### 보안 점검

- **무엇이 문제인지**
  - 기본 비밀번호는 인증을 너무 쉽게 통과하게 만들 수 있습니다.
  - 열린 디버그 포트는 내부 정보와 제어 경로를 노출할 수 있습니다.
  - 오래된 펌웨어는 이미 알려진 취약점이 수정되지 않은 상태일 수 있습니다.

- **왜 위험한지**
  - 참고 자료의 모의해킹 예시에서도 **취약한 비밀번호**, **암호화 미흡**, **암호화 미적용 업데이트**, **펌웨어 버전 출력**, **디버그 포트**, **펌웨어 분석**이 중요한 공격 포인트로 제시됩니다.
  - 즉, 이 세 가지는 서로 따로가 아니라 연결되어 있습니다.  
    기본 비밀번호로 들어가거나, 디버그 포트를 통해 내부를 들여다보거나, 오래된 펌웨어의 취약점을 이용해 장치를 장악할 수 있습니다.

- **실제로 발생할 수 있는 일**
  - 인증되지 않은 사용자가 기기를 제어할 수 있습니다.
  - 펌웨어가 추출되거나 변조될 수 있습니다.
  - 저장된 정보나 설정이 노출될 수 있습니다.
  - 경우에 따라 다른 장치로 공격이 확산될 수도 있습니다.

### 무엇부터 고칠지

1. **기본 비밀번호 즉시 변경 및 기본 계정 제거**
   - 이유: 가장 쉽게 악용되는 진입점입니다.
   - 기본값이 남아 있으면 외부에서 바로 시도할 수 있어 위험이 큽니다.

2. **열린 디버그 포트 비활성화 또는 양산 제품에서 차단**
   - 이유: 기기 내부로 들어가는 직접 통로이기 때문입니다.
   - 분해한 뒤 내부 프로그램이나 설정을 읽는 데 악용될 수 있습니다.

3. **펌웨어를 최신 버전으로 안전하게 업데이트**
   - 이유: 이미 알려진 취약점을 제거해야 합니다.
   - 참고 자료에 따르면 업데이트 파일은 **암호화된 연결로 전송**하고, **서명과 무결성 검증**을 해야 합니다.

4. **업데이트 사실을 사용자에게 알리고 검증 절차 적용**
   - 이유: 사용자가 모르는 사이에 갱신되거나, 변조된 파일이 들어가는 것을 막기 위해서입니다.
   - 가능하면 **보안 부팅**도 고려해야 합니다.

### 통합 개선안

- **개발용 디버그 기능은 양산 제품에서 제거하거나 비활성화**
- **기본 비밀번호는 출하 후 반드시 변경하도록 강제**
- **펌웨어 업데이트는 암호화된 연결과 서명 검증을 함께 적용**
- **업데이트 후 버전과 적용 여부를 확인할 수 있게 구성**
- **모의해킹 관점에서 디버그 포트, 장치 메모리, 펌웨어를 함께 재점검**

### 참고 자료

- IoT_공통보안가이드(최종).pdf, p.57
- IoT_공통보안가이드(최종).pdf, p.58

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합